# ⚡ Arbitrage Calculator: Cloud GPU ML Matcher
This notebook offloads the 474+ million deep semantic cosine similarity matrix calculations from your local CPU to a free **Google Cloud T4 GPU**.

### Instructions
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime > Run all**.
3. The matched arbitrage pairs will be securely posted back to your local `arbitrage-calculator` dashboard.

In [ ]:
# 1. Install required packages
!pip install sentence-transformers torch httpx pydantic -q

import torch
from sentence_transformers import SentenceTransformer, util
import httpx
import asyncio
from typing import List, Dict, Any
import time

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Load the ML Model directly onto the Cloud GPU
print("Loading MiniLM semantic model onto GPU...")
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')
print("Model loaded successfully!")

In [ ]:
# 3. Configure Local Connection (ngrok or local IP)
# If your local backend is exposed via ngrok, put the URL here.
# Otherwise, you can upload the 'markets.json' file directly to Colab.
LOCAL_BACKEND_URL = "http://YOUR_NGROK_URL_HERE/api"

def fetch_markets_from_local():
    try:
        # In a real cloud setup, your local server exposes an endpoint to serve the raw markets
        response = httpx.get(f"{LOCAL_BACKEND_URL}/raw-markets", timeout=30.0)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Could not reach local backend: {e}")
        print("Please upload 'markets.json' to the Colab files pane instead.")
        import json
        import os
        if os.path.exists('markets.json'):
             with open('markets.json', 'r') as f:
                 return json.load(f)
        return []

all_markets = fetch_markets_from_local()
print(f"Fetched {len(all_markets)} markets for processing.")

In [ ]:
# 4. Cloud GPU Matrix Multiplication Logic
def compute_pair_arb(ma, mb):
    # (Simplified for demonstration - replicates backend/matcher.py logic)
    price1_a, price1_b = ma['yesPrice'], 1 - mb['yesPrice']
    price2_a, price2_b = 1 - ma['yesPrice'], mb['yesPrice']
    
    cost1 = price1_a + price1_b
    roi1 = ((1 - cost1) / cost1 * 100) if cost1 < 1 and cost1 > 0 else -100
    
    cost2 = price2_a + price2_b
    roi2 = ((1 - cost2) / cost2 * 100) if cost2 < 1 and cost2 > 0 else -100
    
    if roi1 > roi2:
        return {"roi": roi1, "cost": cost1, "scenario": 1}
    return {"roi": roi2, "cost": cost2, "scenario": 2}

def match_on_gpu(markets, min_similarity=65.0):
    by_platform = {}
    for m in markets:
        if m.get("isBinary", True) and m.get("outcomeCount", 2) == 2:
            by_platform.setdefault(m["platform"], []).append(m)
            
    platforms = list(by_platform.keys())
    print(f"Processing platforms: {platforms}")
    
    # 1. Generate Embeddings on GPU
    start_time = time.time()
    plat_embeddings = {}
    for plat in platforms:
        titles = [m["title"] for m in by_platform[plat]]
        print(f"Encoding {len(titles)} markets for {plat}...")
        plat_embeddings[plat] = model.encode(titles, convert_to_tensor=True, batch_size=256)
        
    # 2. Matrix Multiplication on GPU
    pairs = []
    threshold_tensor = min_similarity / 100.0
    
    print("\nExecuting cross-platform matrix multiplications on GPU...")
    for i in range(len(platforms)):
        pa = platforms[i]
        emb_a = plat_embeddings[pa]
        if emb_a is None: continue
        
        for j in range(i + 1, len(platforms)):
            pb = platforms[j]
            emb_b = plat_embeddings[pb]
            if emb_b is None: continue
            
            # Lightning fast GPU cosine similarity (milliseconds for millions of pairs!)
            cosine_scores = util.cos_sim(emb_a, emb_b)
            
            # Find indices where score > threshold
            high_score_indices = (cosine_scores >= threshold_tensor).nonzero(as_tuple=False)
            
            for idx in high_score_indices:
                idx_a, idx_b = idx[0].item(), idx[1].item()
                score = cosine_scores[idx_a][idx_b].item()
                
                ma = by_platform[pa][idx_a]
                mb = by_platform[pb][idx_b]
                
                arb_data = compute_pair_arb(ma, mb)
                if arb_data["roi"] > -100:
                    pairs.append({
                        "marketA": ma, 
                        "marketB": mb, 
                        "roi": arb_data["roi"],
                        "matchScore": round(score * 100, 1)
                    })
                    
    end_time = time.time()
    print(f"\nGPU Matching Complete! Found {len(pairs)} opportunities in {end_time - start_time:.2f} seconds.")
    return pairs

if all_markets:
    found_pairs = match_on_gpu(all_markets)
else:
    print("No markets loaded to process.")

In [ ]:
# 5. Post Results Back to Local App
def post_results_to_system(pairs):
    if not pairs:
        return
        
    try:
        print(f"Posting {len(pairs)} results back to {LOCAL_BACKEND_URL}...")
        response = httpx.post(f"{LOCAL_BACKEND_URL}/cloud-results", json=pairs)
        response.raise_for_status()
        print("Successfully synced pairs to your local dashboard!")
    except Exception as e:
        print(f"Failed to post back to local system: {e}")
        print("You can also download this cell's output as a JSON file and drag it into your app.")
        
post_results_to_system(found_pairs)